# 4. Check whether a batch looks different

**Question:** Does incoming customer activity look different from the training reference?

Run notebooks 01 and 02 first. Scoring is not required for this notebook.
We compare feature distributions and predicted probabilities using Population Stability Index (PSI).
This is an optional offline extension. It does not measure prediction accuracy without true churn labels.

In [ ]:
import sys
from pathlib import Path

# Jupyter may start in the project root or in notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "source.json").exists():
    raise FileNotFoundError("Open Jupyter from the RetainAI project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

## Load reference and current data

The saved training table is the reference. The default current table is the test holdout, so this is
an offline demonstration. Change INPUT_PATH to inspect a separate batch. Validation uses the same
schema as scoring.

In [ ]:
import json

import numpy as np
import pandas as pd

from retainai.features import CATEGORICAL, NUMERIC, validate_customers
from retainai.scoring import load_model

INPUT_PATH = OUTPUT / "test.csv"
bundle = load_model()
reference = validate_customers(bundle["reference"])
current = validate_customers(pd.read_csv(INPUT_PATH, dtype={"customer_id": str}))
print("Reference rows:", len(reference), "Current rows:", len(current))

## Compare proportions across bins

PSI sums `(current proportion − reference proportion) × log(current proportion / reference proportion)`.
Equal proportions produce zero. Larger differences produce larger values.
Adding 0.5 to each count avoids division by zero. This smoothing also makes results sensitive to batch size,
especially for small samples. The thresholds below are heuristics, not significance tests.

In [ ]:
def distribution_psi(reference_counts, current_counts):
    reference = np.asarray(reference_counts, dtype=float) + 0.5
    current = np.asarray(current_counts, dtype=float) + 0.5
    reference /= reference.sum()
    current /= current.sum()
    return float(np.sum((current - reference) * np.log(current / reference)))

## Bin numeric values

Cut points come only from training quantiles. Infinite outside edges keep values beyond the training
range in the counts. A constant reference gets a separate bin so changed values are still detected.

In [ ]:
def numeric_psi(reference, current):
    reference, current = np.asarray(reference), np.asarray(current)
    if not len(reference) or not len(current):
        raise ValueError("Drift needs two nonempty samples.")
    if not np.isfinite(reference).all() or not np.isfinite(current).all():
        raise ValueError("Drift inputs must be finite.")
    cuts = np.unique(np.quantile(reference, np.linspace(0.1, 0.9, 9)))
    if len(cuts) == 1:
        # Give a constant reference value its own bin so any change is visible.
        cuts = np.r_[cuts, np.nextafter(cuts[0], np.inf)]
    # Infinite outer edges keep values beyond the training range in the counts.
    edges = np.r_[-np.inf, cuts, np.inf]
    return distribution_psi(
        np.histogram(reference, edges)[0], np.histogram(current, edges)[0]
    )

## Count categorical values

Use the union of both sets of categories so an unseen category is counted too.
Input validation still rejects category codes outside this project's documented schema.

In [ ]:
def categorical_psi(reference, current):
    categories = sorted(set(reference) | set(current))
    ref_counts = pd.Series(reference).value_counts().reindex(categories, fill_value=0)
    cur_counts = pd.Series(current).value_counts().reindex(categories, fill_value=0)
    return distribution_psi(ref_counts, cur_counts)

## Produce a small drift report

We use the largest PSI across inputs and predictions: under 0.10 is stable, 0.10 to below 0.25 is watch,
and 0.25 or above is review. These labels only prompt investigation. They never trigger retraining.

In [ ]:
def compare_batches(reference, current, model):
    features = {name: numeric_psi(reference[name], current[name]) for name in NUMERIC}
    features.update(
        {name: categorical_psi(reference[name], current[name]) for name in CATEGORICAL}
    )
    features["prediction"] = numeric_psi(
        model.predict_proba(reference)[:, 1],
        model.predict_proba(current)[:, 1],
    )
    largest = max(features.values())
    return {
        "reference_rows": len(reference),
        "current_rows": len(current),
        "status": "review"
        if largest >= 0.25
        else "watch"
        if largest >= 0.10
        else "stable",
        "psi": {name: round(value, 4) for name, value in features.items()},
        "note": "Heuristic drift signal; investigate before retraining. No automatic deployment or rollback.",
    }

In [ ]:
report = compare_batches(reference, current, bundle["model"])
report["batch_kind"] = (
    "offline holdout demo" if INPUT_PATH == OUTPUT / "test.csv" else "provided batch"
)
(OUTPUT / "drift.json").write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)
display(pd.Series(report["psi"]).sort_values(ascending=False))
print("Status:", report["status"], "— saved output/drift.json")

## Prove that the check reacts to a known change

Identical tables should be stable. A synthetic batch with 1,000 call failures per customer should trigger
review. This deliberately extreme scenario tests the code; it is not evidence of actual customer drift.
The simulation does not overwrite the saved real-batch report.

In [ ]:
assert numeric_psi(np.ones(100), np.ones(100)) == 0
assert numeric_psi(np.ones(100), np.full(100, 2)) > 0.25
assert categorical_psi([1] * 100, [2] * 100) > 0.25
assert compare_batches(reference, reference, bundle["model"])["status"] == "stable"
shifted = current.copy()
shifted["call_failure"] = 1000
simulation = compare_batches(reference, shifted, bundle["model"])
assert simulation["status"] == "review"
print("Synthetic shift status:", simulation["status"])
print("Synthetic call_failure PSI:", simulation["psi"]["call_failure"])

## What happens after review?

Check units, formatting, data collection, and customer mix first. Obtain actual churn labels before
measuring accuracy or deciding whether to retrain. Distribution drift does not prove the model is wrong,
and stable distributions do not prove it is right. You have now completed the offline workflow.